# KnowBase AI - experiments

In [5]:
import os
print(os.getcwd())
print(os.listdir())

c:\Users\MONEEM\NIAI Internship\KnowBase_AI
['.env', '.git', '.gitignore', '.ipynb_checkpoints', '.vscode', 'assets', 'chatbot.ipynb', 'requirements.txt', 'venv']


In [6]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv
api_key = os.getenv("GROQ_API_KEY")
print("Key loaded:", api_key is not None)

client = Groq(api_key=api_key)
MODEL = "qwen/qwen3.8-27b"

Key loaded: True


In [7]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Explain RAG in two sentences."}],
)
print(response.choices[0].message.content)

Retrieval-Augmented Generation (RAG) is an AI framework that enhances large language models by first retrieving relevant external information from a knowledge base before generating a response. This process improves answer accuracy and reduces hallucinations by grounding the output in specific, up-to-date, or proprietary data.


# Extract PDF text

In [8]:
import fitz #PyMuPDF

PDF_PATH = "Moneem Resume.pdf"

doc = fitz.open("C:/Users/MONEEM/NIAI Internship/KnowBase_AI/assets/Muhammad Moneem Resume.pdf")
pages = [(i + 1, page.get_text().strip()) for i, page in enumerate(doc)]
pages = [(num, text) for num, text in pages if text]  # drop empty pages

print("Total pages:", len(doc))
print("Pages with text:", len(pages))
print("\n--- Page", pages[0][0], "preview ---")
print(pages[0][1][:500])

Total pages: 1
Pages with text: 1

--- Page 1 preview ---
Muhammad Moneem Raza 
moneemraxa@gmail.com   |   +923081125201   |   Lahore, Pakistan   |   LinkedIn   |   GitHub   |   Portfolio 
Summary 
Full-stack software developer specializing in Flutter, with a track record of building scalable, cross-platform mobile applications 
backed by Firebase and REST APIs. Experienced in delivering features for a digital banking application, with a focus on clean 
architecture, performance, and reliable user experiences. BS Computer Science graduate, now expandin


c:\Users\MONEEM\NIAI Internship\KnowBase_AI\venv\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


# Chunking

In [12]:
CHUNK_SIZE = 900   # characters per chunk
OVERLAP = 150 # characters shared with the next chunk

def chunk_pages(pages, source):
    chunks = []
    step = CHUNK_SIZE - OVERLAP
    for page_no, text in pages:
        for start in range(0, len(text), step):
            piece = text[start:start + CHUNK_SIZE].strip()
            if len(piece) > 50:
                chunks.append({"text": piece, "source": source, "page": page_no})
    return chunks

chunks = chunk_pages(pages, PDF_PATH)

print("Total chunks:", len(chunks))

# Add this check to prevent the IndexError crash
if len(chunks) > 0:
    print("\nChunk 0:", chunks[0]["source"], "page", chunks[0]["page"])
    print(chunks[0]["text"][:300])
    if len(chunks) > 1:
        print("\nChunk 1 starts with:", chunks[1]["text"][:100])
else:
    print("\n⚠️ No chunks were created. Check if 'pages' contains any text!")
    print("Current 'pages' content:", pages)

Total chunks: 4

Chunk 0: Moneem Resume.pdf page 1
Muhammad Moneem Raza 
moneemraxa@gmail.com   |   +923081125201   |   Lahore, Pakistan   |   LinkedIn   |   GitHub   |   Portfolio 
Summary 
Full-stack software developer specializing in Flutter, with a track record of building scalable, cross-platform mobile applications 
backed by Firebase and REST

Chunk 1 starts with: elancing at Innovista learneasy.pk 
Skills 
Mobile Development: Flutter, Firebase, Firestore 
Tools:


# Embeddings and FAISS

In [14]:
import faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

vectors = embedder.encode(
    [c["text"] for c in chunks],
    normalize_embeddings=True,
).astype("float32")

index = faiss.IndexFlatIP(vectors.shape[1])
index.add(vectors)

print("Vector shape:", vectors.shape)
print("Vectors in index:", index.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector shape: (4, 384)
Vectors in index: 4


In [15]:
#retrival function and test

TOP_K = 4
MIN_SCORE = 0.25

def retrieve(question, k=TOP_K, min_score=MIN_SCORE):
    q = embedder.encode([question], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, min(k, len(chunks)))
    return [
        {**chunks[i], "score": float(s)}
        for s, i in zip(scores[0], ids[0])
        if i != -1 and s >= min_score
    ]

# Change this to a question your PDF can actually answer
question = "What is this document about?"

for hit in retrieve(question):
    print(f"Page {hit['Page']} | score {hit['score']:.2f}")
    print(hit["text"][:200], "\n")